# Cognition without consciousness

Ulric Neisser (1967) defined cognition as the processes by which sensory input is **transformed, reduced, elaborated, stored, recovered, and used**.

This notebook builds a small foraging agent in which each of those six processes is one explicit function, runs it, and then removes processes one at a time to show that each does real work. Nothing here is conscious: it is seeded, deterministic arithmetic over lists and dicts.

The code below is identical to [`bacteria_cog.py`](bacteria_cog.py); the outputs are saved, so you can read the results without running anything.

## Setup

The world is a 30×30 grid that wraps around at the edges. Scent is only detectable within 4 cells of a food patch.

In [1]:
import pandas as pd

import csv
import math
import random

SIZE = 30                     # world is a SIZE x SIZE torus
SMELL_RANGE = 4               # scent is only detectable this close to a patch
DIRS = [(0, 1), (1, 0), (0, -1), (-1, 0), (1, 1), (1, -1), (-1, 1), (-1, -1)]
DIR_NAMES = {(0, 1): "N", (1, 0): "E", (0, -1): "S", (-1, 0): "W",
             (1, 1): "NE", (1, -1): "SE", (-1, 1): "NW", (-1, -1): "SW", (0, 0): "-"}
THRESH = 1.5                  # salience threshold used by reduce(), above the noise floor
REGROW = 0.2                  # food regrowth per patch per step
WORTH_IT = 6.0                # minimum expected regrowth to revisit a patch
STEPS = 400
TRIALS = 60

COLUMNS = ["Step", "Pos", "Transform", "Reduce", "Elaborate", "Store", "Recover", "Use"]


def dist(p, q):
    dx = min(abs(p[0] - q[0]), SIZE - abs(p[0] - q[0]))
    dy = min(abs(p[1] - q[1]), SIZE - abs(p[1] - q[1]))
    return math.hypot(dx, dy)


def wrap(p):
    return (p[0] % SIZE, p[1] % SIZE)


def sign(x):
    return (x > 0) - (x < 0)


## The world

Six food patches that empty when eaten and slowly regrow. The agent never sees this map; it only receives `smell()`: a noisy reading one step away in each of eight directions.

In [2]:
class World:
    """Food patches that deplete when eaten and slowly regrow."""

    def __init__(self, rng, n_patches=6):
        self.rng = rng
        self.food = {}
        while len(self.food) < n_patches:
            self.food[(rng.randrange(SIZE), rng.randrange(SIZE))] = 10.0

    def smell(self, pos):
        """Raw sensory input: noisy, short-range scent one step away in 8 directions."""
        raw = []
        for dx, dy in DIRS:
            p = wrap((pos[0] + dx, pos[1] + dy))
            s = sum(v / (1 + dist(p, q) ** 2) for q, v in self.food.items()
                    if dist(p, q) <= SMELL_RANGE)
            raw.append(s + self.rng.gauss(0, 0.2))
        return raw

    def eat(self, pos):
        v = self.food.get(pos, 0.0)
        if pos in self.food:
            self.food[pos] = 0.0
        return v

    def regrow(self):
        for q in self.food:
            self.food[q] = min(10.0, self.food[q] + REGROW)


## The agent: six processes

Each method is one of Neisser's processes, and each returns a short description so we can watch it work.

In [3]:
class Agent:
    """Each process returns its result plus a short description for the trace table."""

    def __init__(self, rng, use_filter=True, use_memory=True):
        self.rng = rng
        self.pos = (0, 0)
        self.memory = {}          # patch location -> time last harvested
        self.use_filter = use_filter
        self.use_memory = use_memory

    # 1. TRANSFORM: raw intensities -> egocentric gradient vector
    def transform(self, raw):
        mean = sum(raw) / len(raw)
        gx = sum((r - mean) * dx for r, (dx, dy) in zip(raw, DIRS))
        gy = sum((r - mean) * dy for r, (dx, dy) in zip(raw, DIRS))
        return (gx, gy), f"grad ({gx:+.2f}, {gy:+.2f})"

    # 2. REDUCE: attention; discard signals too weak to trust
    def reduce(self, grad):
        mag = math.hypot(*grad)
        if not self.use_filter:
            return grad, f"filter off |g|={mag:.2f}"
        if mag > THRESH:
            return grad, f"kept |g|={mag:.2f}"
        return None, f"dropped |g|={mag:.2f}"

    # 3. ELABORATE: go beyond the input; infer what it implies for the agent
    def elaborate(self, signal, bite):
        if bite >= 0.5:
            return ("food here", None), f"food here ({bite:.1f})"
        if signal is not None:
            gx, gy = signal
            heading = max(DIRS, key=lambda d: (d[0] * gx + d[1] * gy) / math.hypot(*d))
            return ("food lies toward", heading), f"food lies {DIR_NAMES[heading]}"
        return ("input uninformative", None), "uninformative"

    # 4. STORE: record where food was found and when it was last harvested
    def store(self, bite, t):
        if not self.use_memory:
            return "memory off"
        if bite > 0 or self.pos in self.memory:
            action = "updated" if self.pos in self.memory else "new"
            self.memory[self.pos] = t
            return f"{action} {self.pos}@t{t}"
        return "-"

    # 5. RECOVER: cued by uninformative input, retrieve the remembered patch
    #    expected to have regrown the most, discounted by distance
    def recover(self, signal, t):
        if not self.use_memory:
            return None, "memory off"
        if signal is not None:
            return None, "not cued"
        if not self.memory:
            return None, "memory empty"

        def expected(k):
            regrown = min(10.0, (t - self.memory[k]) * REGROW)
            return regrown / (1 + dist(self.pos, k))

        # only worth recalling a patch expected to have regrown substantially
        candidates = [k for k in self.memory if k != self.pos
                      and (t - self.memory[k]) * REGROW >= WORTH_IT]
        if not candidates:
            return None, f"none worth it ({len(self.memory)} known)"
        best = max(candidates, key=expected)
        regrown = min(10.0, (t - self.memory[best]) * REGROW)
        return best, f"{best} expect ~{regrown:.1f}"

    # 6. USE: select an action among alternatives
    def use(self, meaning, target):
        kind, heading = meaning
        if kind == "food here":
            return (0, 0), "stay & eat"
        if kind == "food lies toward":
            return heading, f"follow scent {DIR_NAMES[heading]}"
        if target is not None:
            dx = (target[0] - self.pos[0] + SIZE // 2) % SIZE - SIZE // 2
            dy = (target[1] - self.pos[1] + SIZE // 2) % SIZE - SIZE // 2
            move = (sign(dx), sign(dy))
            return move, f"go to memory {DIR_NAMES[move]}"
        move = self.rng.choice(DIRS)
        return move, f"explore {DIR_NAMES[move]}"

    def step(self, world, t):
        """Run all six processes once; return food eaten and a row for the trace table."""
        start = self.pos
        bite = world.eat(self.pos)
        raw = world.smell(self.pos)
        grad, d_transform = self.transform(raw)
        signal, d_reduce = self.reduce(grad)
        meaning, d_elaborate = self.elaborate(signal, bite)
        d_store = self.store(bite, t)
        target, d_recover = self.recover(signal, t)
        move, d_use = self.use(meaning, target)
        self.pos = wrap((self.pos[0] + move[0], self.pos[1] + move[1]))
        row = [str(t), str(start), d_transform, d_reduce, d_elaborate, d_store, d_recover, d_use]
        return bite, row


## A baseline with no cognition

A random walker that eats when it lands on food and otherwise wanders.

In [4]:
class RandomWalker:
    """Baseline: no processing of sensory input at all."""

    def __init__(self, rng):
        self.rng = rng
        self.pos = (0, 0)

    def step(self, world, t):
        bite = world.eat(self.pos)
        if bite < 0.5:
            dx, dy = self.rng.choice(DIRS)
            self.pos = wrap((self.pos[0] + dx, self.pos[1] + dy))
        return bite, None


## Watching the processes

One row per step, one column per process, for 100 steps in a seeded world.

In [5]:
rng = random.Random(1)
world, agent = World(rng), Agent(rng)
rows = []
for t in range(100):
    _, row = agent.step(world, t)
    rows.append(row)
    world.regrow()

trace = pd.DataFrame(rows, columns=COLUMNS).set_index("Step")
pd.set_option("display.max_rows", 200)
trace

,Pos,Transform,Reduce,Elaborate,Store,Recover,Use
Step,,,,,,,
0,"(0, 0)","grad (-0.38, +0.06)",dropped |g|=0.38,uninformative,-,memory empty,explore SW
1,"(29, 29)","grad (-0.44, -1.63)",kept |g|=1.69,food lies S,-,not cued,follow scent S
2,"(29, 28)","grad (-2.54, -3.53)",kept |g|=4.35,food lies SW,-,not cued,follow scent SW
3,"(28, 27)","grad (-3.84, -6.68)",kept |g|=7.71,food lies SW,-,not cued,follow scent SW
4,"(27, 26)","grad (-0.31, -13.73)",kept |g|=13.74,food lies S,-,not cued,follow scent S
5,"(27, 25)","grad (+0.91, -1.00)",dropped |g|=1.35,food here (10.0),"new (27, 25)@t5",none worth it (1 known),stay & eat
6,"(27, 25)","grad (+0.10, -0.98)",dropped |g|=0.98,uninformative,"updated (27, 25)@t6",none worth it (1 known),explore SE
7,"(28, 24)","grad (+0.02, +0.87)",dropped |g|=0.87,uninformative,-,none worth it (1 known),explore NE
8,"(29, 25)","grad (-0.77, +0.29)",dropped |g|=0.82,uninformative,-,none worth it (1 known),explore W


Two steps worth looking at. At step 46 the agent finds a patch and stores it. At step 72 the scent is too weak to use (Reduce drops it), so Recover is cued: the agent heads for a remembered patch it predicts has regrown, though it cannot currently smell it.

In [6]:
trace.loc[["46", "72"]]

,Pos,Transform,Reduce,Elaborate,Store,Recover,Use
Step,,,,,,,
46,"(24, 2)","grad (-0.45, -0.83)",dropped |g|=0.94,food here (10.0),"new (24, 2)@t46",none worth it (2 known),stay & eat
72,"(24, 2)","grad (+0.02, -0.87)",dropped |g|=0.87,uninformative,"updated (24, 2)@t72","(27, 25) expect ~6.6",go to memory SE


## Ablation: does each process do real work?

Mean food collected over 400 steps across 60 seeded worlds, for the full agent, the agent with a process removed, and the random baseline.

In [7]:
def run(make_agent, seed):
    rng = random.Random(seed)
    world = World(rng)
    agent = make_agent(rng)
    total = 0.0
    for t in range(STEPS):
        bite, _ = agent.step(world, t)
        total += bite
        world.regrow()
    return total


conditions = {
    "Random walker (no processing)": lambda r: RandomWalker(r),
    "Full agent (all six processes)": lambda r: Agent(r),
    "Without Reduce (no attention filter)": lambda r: Agent(r, use_filter=False),
    "Without Store/Recover (no memory)": lambda r: Agent(r, use_memory=False),
}
results = pd.DataFrame(
    {"Mean food": [sum(run(make, s) for s in range(TRIALS)) / TRIALS for make in conditions.values()]},
    index=list(conditions),
).round(1)
results

,Mean food
Random walker (no processing),16.2
Full agent (all six processes),210.6
Without Reduce (no attention filter),147.0
Without Store/Recover (no memory),140.4


Removing the attention filter or memory costs the agent roughly a third of its food, and the random walker collects less than a tenth as much. Every process is an explicit function and none involves awareness, so this is cognitive work in Neisser's sense without consciousness.

## What this shows, and what it doesn't

It shows that Neisser's definition can be fully satisfied without consciousness. It can't show that the definition is the right one; for that, the stronger evidence is human cognition without awareness, such as blindsight, priming, implicit learning, and skilled action. The agent is deliberately minimal: hand-set parameters, no world model it revises, and an assigned goal.